In [3]:
!pip install tensorflow

In [4]:
# ============================================================
# 11_final_test_2024_2025_winners_eval.ipynb
#
# Final locked evaluation on 2024-latest available data
#
# Selected procedures:
#   1) autoARIMA
#   2) SARIMA(1,0,0)x(1,0,0)[52] (initial)
#   3) RandomForest(lags=4)
#   4) rf_global_all_infections_all_countries
#   5) DL_GlobalGRU_all_infections_all_countries
#
# Additional procedures:
#   6) ensemble_mean_5
#   7) ensemble_median_5
#
# Methodology:
#   - Data: all dated incidence files in /data
#   - Train: 2014-01-05 to 2023-12-31
#   - Final test: 2024-01-07 to latest available observed week
#   - Horizons: 1,2,3,4
#   - Fixed countries, same as validated 2023 benchmark:
#       ARI: BE, BG, CZ, DE, EE, LT, RO, SI
#       ILI: BE, CZ, EE, GR, IE, LT, NL, PL, RO, SI
# ============================================================

# ============================================================
# 0. Imports
# ============================================================

import sys
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    display = print

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_project_paths
from src.impute import impute_series_weekly
from src.metrics import mae, rmse, mase_scale_seasonal, mase_from_mae

from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    from pmdarima.arima import auto_arima
    PMDARIMA_OK = True
except Exception:
    PMDARIMA_OK = False

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models, callbacks, optimizers, losses
    TF_OK = True
except Exception:
    TF_OK = False


# ============================================================
# 1. Configuration
# ============================================================

paths = get_project_paths(PROJECT_ROOT)

DATA_DIR = paths.data
RESULTS_DIR = paths.results
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FINAL_OUT_DIR = RESULTS_DIR / "final_test_2024_2025"
FINAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_START = pd.Timestamp("2014-01-05")
TRAIN_END = pd.Timestamp("2023-12-31")
TEST_START = pd.Timestamp("2024-01-07")

WEEK_FREQ = "W-SUN"

M_SEASON = 52
HORIZONS = (1, 2, 3, 4)
MAX_H = max(HORIZONS)

OK_ARI = ["BE", "BG", "CZ", "DE", "EE", "LT", "RO", "SI"]
OK_ILI = ["BE", "CZ", "EE", "GR", "IE", "LT", "NL", "PL", "RO", "SI"]

COMMON6 = ["BE", "CZ", "EE", "LT", "RO", "SI"]

RUN_AUTOARIMA = True
RUN_DL = True

if RUN_AUTOARIMA and not PMDARIMA_OK:
    raise ImportError(
        "pmdarima is not installed, but RUN_AUTOARIMA=True. "
        "Install it with: pip install pmdarima"
    )

if RUN_DL and not TF_OK:
    raise ImportError(
        "TensorFlow is not installed, but RUN_DL=True. "
        "Run this notebook in Colab/GPU or install TensorFlow locally."
    )

SEED = 42
np.random.seed(SEED)

if TF_OK:
    tf.random.set_seed(SEED)

SARIMA_INITIAL_NAME = "SARIMA(1,0,0)x(1,0,0)[52] (initial)"
AUTOARIMA_NAME = "autoARIMA"
RF_LOCAL_NAME = "RandomForest(lags=4)"
RF_POOLED_NAME = "rf_global_all_infections_all_countries"
DL_GRU_NAME = "DL_GlobalGRU_all_infections_all_countries"

ENSEMBLE_MEAN_NAME = "ensemble_mean_5"
ENSEMBLE_MEDIAN_NAME = "ensemble_median_5"

SELECTED_BASE_MODELS = [
    AUTOARIMA_NAME,
    SARIMA_INITIAL_NAME,
    RF_LOCAL_NAME,
    RF_POOLED_NAME,
    DL_GRU_NAME,
]


# ============================================================
# 2. Load all dated data snapshots
# ============================================================

def standardize_incidence_df(df):
    df = df.copy()

    possible_location_cols = ["location", "country", "geo_value", "geo_id"]
    possible_date_cols = ["truth_date", "date", "week", "time_value", "target_end_date"]
    possible_value_cols = ["value", "incidence", "rate", "target", "y"]

    location_col = next((c for c in possible_location_cols if c in df.columns), None)
    date_col = next((c for c in possible_date_cols if c in df.columns), None)
    value_col = next((c for c in possible_value_cols if c in df.columns), None)

    if location_col is None or date_col is None or value_col is None:
        raise ValueError(
            f"Could not infer columns. Available columns: {list(df.columns)}"
        )

    out = df[[location_col, date_col, value_col]].copy()
    out.columns = ["location", "truth_date", "value"]

    out["truth_date"] = pd.to_datetime(out["truth_date"])
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out["location"] = out["location"].astype(str)

    return out


def load_dated_snapshots(data_dir: Path, disease: str):
    """
    Loads files like:
      2024-10-11-ARI_incidence.csv
      2025-12-19-ILI_incidence.csv

    If the same (location, truth_date) appears in several snapshots,
    the value from the latest snapshot file is kept.
    """

    pattern = re.compile(rf"(\d{{4}}-\d{{2}}-\d{{2}})-{disease}_incidence\.csv$", re.IGNORECASE)

    files = []

    for p in data_dir.glob(f"*-{disease}_incidence.csv"):
        m = pattern.search(p.name)

        if m:
            snapshot_date = pd.to_datetime(m.group(1))
            files.append((snapshot_date, p))

    if len(files) == 0:
        fallback = data_dir / f"latest-{disease}_incidence.csv"
        if fallback.exists():
            files = [(pd.Timestamp("2099-01-01"), fallback)]
        else:
            raise FileNotFoundError(f"No dated or latest file found for {disease} in {data_dir}")

    parts = []

    for snapshot_date, path in sorted(files):
        df = pd.read_csv(path)
        df = standardize_incidence_df(df)
        df["snapshot_date"] = snapshot_date
        df["source_file"] = path.name
        parts.append(df)

    all_df = pd.concat(parts, ignore_index=True)

    all_df = (
        all_df
        .sort_values(["location", "truth_date", "snapshot_date"])
        .drop_duplicates(["location", "truth_date"], keep="last")
        .sort_values(["location", "truth_date"])
        .reset_index(drop=True)
    )

    return all_df, pd.DataFrame(
        [{"snapshot_date": s, "file": str(p.name)} for s, p in sorted(files)]
    )


ari_df, ari_files_used = load_dated_snapshots(DATA_DIR, "ARI")
ili_df, ili_files_used = load_dated_snapshots(DATA_DIR, "ILI")

print("ARI loaded:", ari_df.shape)
print("ILI loaded:", ili_df.shape)

print("\nARI files used:")
display(ari_files_used.head())
display(ari_files_used.tail())

print("\nILI files used:")
display(ili_files_used.head())
display(ili_files_used.tail())

print("\nARI date range:", ari_df["truth_date"].min(), "->", ari_df["truth_date"].max())
print("ILI date range:", ili_df["truth_date"].min(), "->", ili_df["truth_date"].max())


# ============================================================
# 3. Calendars and series helpers
# ============================================================

max_available_date = max(ari_df["truth_date"].max(), ili_df["truth_date"].max())

full_weeks = pd.date_range(
    start=TRAIN_START,
    end=max_available_date,
    freq=WEEK_FREQ
)

train_weeks = full_weeks[
    (full_weeks >= TRAIN_START) &
    (full_weeks <= TRAIN_END)
]

test_weeks = full_weeks[
    (full_weeks >= TEST_START) &
    (full_weeks <= max_available_date)
]

print("\nTrain weeks:", len(train_weeks), train_weeks.min(), "->", train_weeks.max())
print("Test weeks :", len(test_weeks), test_weeks.min(), "->", test_weeks.max())


def extract_raw_weekly_series(df, location, calendar):
    dfloc = df[df["location"] == location].copy()
    dfloc["truth_date"] = pd.to_datetime(dfloc["truth_date"])

    if len(dfloc) == 0:
        return pd.Series(index=calendar, dtype=float)

    s = (
        dfloc
        .sort_values("truth_date")
        .drop_duplicates("truth_date", keep="last")
        .set_index("truth_date")["value"]
        .astype(float)
        .reindex(calendar)
    )

    return s


def get_train_series(df_raw, location):
    return impute_series_weekly(
        df_raw[df_raw["location"] == location][["truth_date", "value"]],
        calendar=train_weeks,
        trim_to_first_obs=True,
        max_gap_knn=8
    ).astype(float)


def get_raw_full_series(df_raw, location):
    return extract_raw_weekly_series(df_raw, location, full_weeks).astype(float)


def get_y_test(df_raw, location):
    return extract_raw_weekly_series(df_raw, location, test_weeks).astype(float)


def mae_np(y, pred):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(pred))))


def rmse_np(y, pred):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(pred)) ** 2)))


def expected_points_from_truth(y_test, disease, location):
    rows = []

    for h in HORIZONS:
        n_expected = 0

        for origin in test_weeks:
            target = origin + pd.Timedelta(weeks=h)

            if target not in y_test.index:
                continue

            if pd.notna(y_test.loc[target]):
                n_expected += 1

        rows.append({
            "disease": disease,
            "location": location,
            "h": int(h),
            "expected_actual_points": int(n_expected)
        })

    return pd.DataFrame(rows)


# ============================================================
# 4. Fixed country validation
# ============================================================

available_ari = sorted(ari_df["location"].dropna().unique())
available_ili = sorted(ili_df["location"].dropna().unique())

missing_ari = sorted(set(OK_ARI) - set(available_ari))
missing_ili = sorted(set(OK_ILI) - set(available_ili))

if missing_ari:
    raise ValueError(f"Missing ARI fixed countries in loaded data: {missing_ari}")

if missing_ili:
    raise ValueError(f"Missing ILI fixed countries in loaded data: {missing_ili}")

print("\nUsing fixed countries:")
print("ARI:", OK_ARI)
print("ILI:", OK_ILI)


# ============================================================
# 5. Expected points and MASE scales
# ============================================================

expected_parts = []
scale_rows = []

for disease, df_raw, locations in [
    ("ARI", ari_df, OK_ARI),
    ("ILI", ili_df, OK_ILI),
]:
    for loc in locations:
        train_s = get_train_series(df_raw, loc)
        y_test = get_y_test(df_raw, loc)

        scale = mase_scale_seasonal(train_s, m=M_SEASON)

        scale_rows.append({
            "disease": disease,
            "location": loc,
            "mase_scale": float(scale)
        })

        expected_parts.append(
            expected_points_from_truth(y_test, disease, loc)
        )

expected_points = pd.concat(expected_parts, ignore_index=True)
mase_scales = pd.DataFrame(scale_rows)

print("\nExpected points aggregated:")
display(
    expected_points
    .groupby(["disease", "h"], as_index=False)
    .agg(
        expected_actual_points=("expected_actual_points", "sum"),
        n_countries=("location", "nunique")
    )
)

print("\nMASE scales:")
display(mase_scales.head())


# ============================================================
# 6. SARIMA initial and autoARIMA
# ============================================================

def fit_sarimax_once(history_init, order, seasonal_order, trend):
    y = pd.Series(history_init).astype(float).copy()

    model = SARIMAX(
        y,
        order=order,
        seasonal_order=seasonal_order,
        trend=trend,
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    res = model.fit(disp=False)
    converged = bool(res.mle_retvals.get("converged", False))

    return res, converged


def pick_autoarima_orders_once(y_train, m=52):
    if not PMDARIMA_OK:
        return None

    y = pd.Series(y_train).dropna().astype(float)

    model = auto_arima(
        y,
        seasonal=True,
        m=m,
        start_p=0,
        start_q=0,
        max_p=2,
        max_q=2,
        start_P=0,
        start_Q=0,
        max_P=1,
        max_Q=1,
        d=None,
        D=None,
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore",
        information_criterion="aic"
    )

    return model.order, model.seasonal_order


def rolling_sarima_eval(
    history_init,
    y_test,
    disease,
    location,
    model_name,
    order,
    seasonal_order,
    trend
):
    rows = []

    try:
        res, converged = fit_sarimax_once(
            history_init=history_init,
            order=order,
            seasonal_order=seasonal_order,
            trend=trend
        )
        fit_ok = True
        fit_error = ""
    except Exception as e:
        res = None
        converged = False
        fit_ok = False
        fit_error = str(e)[:300]

    for origin in test_weeks:
        if res is not None:
            y_obs = y_test.loc[origin] if origin in y_test.index else np.nan
            obs_value = float(y_obs) if pd.notna(y_obs) else np.nan

            try:
                res = res.append([obs_value], refit=False)
            except Exception:
                pass

            try:
                y_hat = res.forecast(steps=MAX_H)
            except Exception:
                y_hat = pd.Series(np.repeat(np.nan, MAX_H))
        else:
            y_hat = pd.Series(np.repeat(np.nan, MAX_H))

        for h in HORIZONS:
            target = origin + pd.Timedelta(weeks=h)

            if target not in y_test.index:
                continue

            actual = y_test.loc[target]

            try:
                pred = y_hat.iloc[h - 1]
            except Exception:
                pred = np.nan

            rows.append({
                "origin": origin,
                "target": target,
                "disease": disease,
                "location": location,
                "h": int(h),
                "y": float(actual) if pd.notna(actual) else np.nan,
                "pred": float(pred) if pd.notna(pred) else np.nan,
                "model": model_name,
                "fit_ok": int(fit_ok),
                "converged": int(converged),
                "fit_error": fit_error,
                "selected_order": str(order),
                "selected_seasonal_order": str(seasonal_order),
                "trend": trend
            })

    return pd.DataFrame(rows)


sarima_long_parts = []
model_status_rows = []

for disease, df_raw, locations in [
    ("ARI", ari_df, OK_ARI),
    ("ILI", ili_df, OK_ILI),
]:
    for loc in locations:
        print(f"\nSARIMA / autoARIMA final test: {disease} {loc}")

        train_s = get_train_series(df_raw, loc)
        y_test = get_y_test(df_raw, loc)

        # SARIMA initial
        df_sarima = rolling_sarima_eval(
            history_init=train_s,
            y_test=y_test,
            disease=disease,
            location=loc,
            model_name=SARIMA_INITIAL_NAME,
            order=(1, 0, 0),
            seasonal_order=(1, 0, 0, 52),
            trend="c"
        )
        sarima_long_parts.append(df_sarima)

        model_status_rows.append({
            "model": SARIMA_INITIAL_NAME,
            "disease": disease,
            "location": loc,
            "selected_order": "(1, 0, 0)",
            "selected_seasonal_order": "(1, 0, 0, 52)",
            "trend": "c",
            "fit_ok": int(df_sarima["fit_ok"].max()) if len(df_sarima) else 0,
            "converged": int(df_sarima["converged"].max()) if len(df_sarima) else 0,
            "fit_error": df_sarima["fit_error"].iloc[0] if len(df_sarima) else "no rows"
        })

        # autoARIMA
        if RUN_AUTOARIMA:
            try:
                order, sorder = pick_autoarima_orders_once(train_s, m=M_SEASON)

                df_auto = rolling_sarima_eval(
                    history_init=train_s,
                    y_test=y_test,
                    disease=disease,
                    location=loc,
                    model_name=AUTOARIMA_NAME,
                    order=order,
                    seasonal_order=sorder,
                    trend="n"
                )

                sarima_long_parts.append(df_auto)

                model_status_rows.append({
                    "model": AUTOARIMA_NAME,
                    "disease": disease,
                    "location": loc,
                    "selected_order": str(order),
                    "selected_seasonal_order": str(sorder),
                    "trend": "n",
                    "fit_ok": int(df_auto["fit_ok"].max()) if len(df_auto) else 0,
                    "converged": int(df_auto["converged"].max()) if len(df_auto) else 0,
                    "fit_error": df_auto["fit_error"].iloc[0] if len(df_auto) else "no rows"
                })

            except Exception as e:
                print(f"autoARIMA failed for {disease} {loc}: {e}")
                model_status_rows.append({
                    "model": AUTOARIMA_NAME,
                    "disease": disease,
                    "location": loc,
                    "selected_order": "",
                    "selected_seasonal_order": "",
                    "trend": "n",
                    "fit_ok": 0,
                    "converged": 0,
                    "fit_error": str(e)[:300]
                })

sarima_long = pd.concat(sarima_long_parts, ignore_index=True)


# ============================================================
# 7. Local Random Forest(lags=4)
# ============================================================

def make_lagged_supervised(series, n_lags=4):
    s = pd.Series(series).astype(float).dropna().reset_index(drop=True)

    X, y = [], []

    for i in range(n_lags, len(s)):
        X.append(s.iloc[i - n_lags:i].values[::-1])
        y.append(s.iloc[i])

    if len(X) == 0:
        return np.empty((0, n_lags)), np.empty((0,))

    return np.asarray(X, dtype=float), np.asarray(y, dtype=float)


def recursive_rf_forecast(model, history, max_h=4, n_lags=4):
    hist = pd.Series(history).astype(float).dropna().tolist()

    if len(hist) < n_lags:
        return np.repeat(np.nan, max_h)

    preds = []
    temp_hist = hist.copy()

    for _ in range(max_h):
        x = np.asarray(temp_hist[-n_lags:][::-1], dtype=float).reshape(1, -1)
        yhat = float(model.predict(x)[0])
        preds.append(yhat)
        temp_hist.append(yhat)

    return np.asarray(preds, dtype=float)


def rolling_rf_local_eval(history_init, y_test, disease, location, n_lags=4):
    history = history_init.astype(float).copy()
    rows = []

    for origin in test_weeks:
        if origin in y_test.index and pd.notna(y_test.loc[origin]):
            history.loc[origin] = float(y_test.loc[origin])

        hist_non_na = history.dropna()

        if len(hist_non_na) <= n_lags:
            preds = np.repeat(np.nan, MAX_H)
        else:
            X_train, y_train = make_lagged_supervised(hist_non_na, n_lags=n_lags)

            if len(X_train) == 0:
                preds = np.repeat(np.nan, MAX_H)
            else:
                try:
                    model = RandomForestRegressor(
                        n_estimators=300,
                        max_depth=None,
                        min_samples_split=4,
                        min_samples_leaf=2,
                        random_state=42,
                        n_jobs=-1
                    )
                    model.fit(X_train, y_train)
                    preds = recursive_rf_forecast(model, hist_non_na, max_h=MAX_H, n_lags=n_lags)
                except Exception:
                    preds = np.repeat(np.nan, MAX_H)

        for h in HORIZONS:
            target = origin + pd.Timedelta(weeks=h)

            if target not in y_test.index:
                continue

            actual = y_test.loc[target]

            rows.append({
                "origin": origin,
                "target": target,
                "disease": disease,
                "location": location,
                "h": int(h),
                "y": float(actual) if pd.notna(actual) else np.nan,
                "pred": float(preds[h - 1]) if pd.notna(preds[h - 1]) else np.nan,
                "model": RF_LOCAL_NAME
            })

    return pd.DataFrame(rows)


rf_local_parts = []

for disease, df_raw, locations in [
    ("ARI", ari_df, OK_ARI),
    ("ILI", ili_df, OK_ILI),
]:
    for loc in locations:
        print(f"\nRF local final test: {disease} {loc}")

        train_s = get_train_series(df_raw, loc)
        y_test = get_y_test(df_raw, loc)

        rf_local_parts.append(
            rolling_rf_local_eval(train_s, y_test, disease, loc, n_lags=4)
        )

rf_local_long = pd.concat(rf_local_parts, ignore_index=True)


# ============================================================
# 8. Pooled global RF
# ============================================================

RECENT_LAGS = [0, 1, 2, 3]
SEASONAL_LAGS = [52, 53, 54, 55]

def make_week_features_single(date):
    week = min(int(date.isocalendar().week), 52)
    return {
        "sin_week": np.sin(2 * np.pi * week / 52.0),
        "cos_week": np.cos(2 * np.pi * week / 52.0)
    }


def make_rf_pooled_training_row(series, t, disease, location):
    row = {
        "disease": disease,
        "location": location
    }

    for lag in RECENT_LAGS:
        row[f"recent_{lag}"] = float(series.iloc[t - lag])

    for lag in SEASONAL_LAGS:
        row[f"seasonal_{lag}"] = float(series.iloc[t - lag])

    row.update(make_week_features_single(series.index[t]))

    for h in HORIZONS:
        row[f"y_{h}"] = float(series.iloc[t + h])

    return row


def build_rf_pooled_training_set():
    rows = []
    min_required_lag = max(SEASONAL_LAGS)

    for disease, df_raw, locations in [
        ("ARI", ari_df, OK_ARI),
        ("ILI", ili_df, OK_ILI),
    ]:
        for loc in locations:
            s = get_train_series(df_raw, loc).dropna()

            for t in range(min_required_lag, len(s) - MAX_H):
                try:
                    rows.append(make_rf_pooled_training_row(s, t, disease, loc))
                except Exception:
                    continue

    return pd.DataFrame(rows).dropna().reset_index(drop=True)


def get_value_available_ffill(available_series, date):
    if date not in available_series.index:
        return np.nan

    val = available_series.loc[date]

    if pd.notna(val):
        return float(val)

    hist = available_series.loc[:date].dropna()

    if len(hist) == 0:
        return np.nan

    return float(hist.iloc[-1])


def make_rf_pooled_eval_row(available_series, origin, disease, location):
    row = {
        "disease": disease,
        "location": location
    }

    for lag in RECENT_LAGS:
        row[f"recent_{lag}"] = get_value_available_ffill(
            available_series,
            origin - pd.Timedelta(weeks=lag)
        )

    for lag in SEASONAL_LAGS:
        row[f"seasonal_{lag}"] = get_value_available_ffill(
            available_series,
            origin - pd.Timedelta(weeks=lag)
        )

    row.update(make_week_features_single(origin))

    return row


rf_pooled_train = build_rf_pooled_training_set()

print("\nRF pooled train shape:", rf_pooled_train.shape)

target_cols = [f"y_{h}" for h in HORIZONS]

rf_pooled_train_enc = pd.get_dummies(
    rf_pooled_train,
    columns=["disease", "location"],
    drop_first=False
)

feature_cols = [c for c in rf_pooled_train_enc.columns if c not in target_cols]

X_train = rf_pooled_train_enc[feature_cols].copy()
y_train = rf_pooled_train_enc[target_cols].copy()

rf_pooled_scaler = StandardScaler()
X_train_scaled = rf_pooled_scaler.fit_transform(X_train)

rf_pooled_model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=400,
        max_depth=12,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
)

rf_pooled_model.fit(X_train_scaled, y_train)

print("RF pooled training complete.")


def pooled_rf_rolling_eval(df_raw, locations, disease_name):
    rows = []

    for loc in locations:
        print(f"\nRF pooled final test: {disease_name} {loc}")

        train_s = get_train_series(df_raw, loc)
        raw_full = get_raw_full_series(df_raw, loc)
        y_test = raw_full.reindex(test_weeks)

        available = pd.Series(index=full_weeks, dtype=float)
        available.loc[train_s.index] = train_s.values

        for origin in test_weeks:
            if origin in y_test.index and pd.notna(y_test.loc[origin]):
                available.loc[origin] = float(y_test.loc[origin])

            feat = make_rf_pooled_eval_row(available, origin, disease_name, loc)
            X_row = pd.DataFrame([feat])

            numeric_cols = [c for c in X_row.columns if c not in ["disease", "location"]]

            if X_row[numeric_cols].isna().any(axis=None):
                preds = np.repeat(np.nan, MAX_H)
            else:
                X_row = pd.get_dummies(X_row, columns=["disease", "location"], drop_first=False)
                X_row = X_row.reindex(columns=feature_cols, fill_value=0)

                try:
                    X_scaled = rf_pooled_scaler.transform(X_row)
                    preds = rf_pooled_model.predict(X_scaled)[0]
                except Exception:
                    preds = np.repeat(np.nan, MAX_H)

            for h in HORIZONS:
                target = origin + pd.Timedelta(weeks=h)

                if target not in y_test.index:
                    continue

                actual = y_test.loc[target]

                rows.append({
                    "origin": origin,
                    "target": target,
                    "disease": disease_name,
                    "location": loc,
                    "h": int(h),
                    "y": float(actual) if pd.notna(actual) else np.nan,
                    "pred": float(preds[h - 1]) if pd.notna(preds[h - 1]) else np.nan,
                    "model": RF_POOLED_NAME
                })

    return pd.DataFrame(rows)


rf_pooled_long = pd.concat(
    [
        pooled_rf_rolling_eval(ari_df, OK_ARI, "ARI"),
        pooled_rf_rolling_eval(ili_df, OK_ILI, "ILI")
    ],
    ignore_index=True
)


# ============================================================
# 9. DL Global GRU
# ============================================================

SEQ_LEN = 104
BATCH_SIZE = 64
EPOCHS_DL = 45
PATIENCE_DL = 6

DISEASE_TO_ID = {"ARI": 0, "ILI": 1}
ALL_LOCATIONS = sorted(set(OK_ARI).union(set(OK_ILI)))
LOCATION_TO_ID = {loc: i for i, loc in enumerate(ALL_LOCATIONS)}

SERIES_DATA = {}

for disease, df_raw, locations in [
    ("ARI", ari_df, OK_ARI),
    ("ILI", ili_df, OK_ILI)
]:
    for loc in locations:
        train_s = get_train_series(df_raw, loc)
        raw_full = get_raw_full_series(df_raw, loc)

        mean = float(train_s.mean())
        std = float(train_s.std(ddof=0))

        if not np.isfinite(std) or std <= 1e-8:
            std = 1.0

        SERIES_DATA[(disease, loc)] = {
            "disease": disease,
            "location": loc,
            "train": train_s,
            "raw_full": raw_full,
            "mean": mean,
            "std": std,
            "disease_id": DISEASE_TO_ID[disease],
            "location_id": LOCATION_TO_ID[loc]
        }

ALL_KEYS = sorted(SERIES_DATA.keys())


def make_week_features_array(dates):
    weeks = np.array([min(int(d.isocalendar().week), 52) for d in dates], dtype=float)
    return (
        np.sin(2 * np.pi * weeks / 52.0).astype("float32"),
        np.cos(2 * np.pi * weeks / 52.0).astype("float32")
    )


def sequence_features(values, dates, mean, std):
    values = pd.Series(values).astype(float)

    if values.isna().any():
        return None

    y_norm = ((values.values - mean) / std).astype("float32")
    sin_week, cos_week = make_week_features_array(dates)

    return np.stack([y_norm, sin_week, cos_week], axis=1).astype("float32")


def build_dl_train_samples(keys):
    X_seq, Y, D_ids, L_ids = [], [], [], []

    for key in keys:
        info = SERIES_DATA[key]
        s = info["train"].dropna().astype(float)
        mean = info["mean"]
        std = info["std"]

        if len(s) < SEQ_LEN + MAX_H + 5:
            continue

        for i in range(SEQ_LEN - 1, len(s) - MAX_H):
            seq = s.iloc[i - SEQ_LEN + 1:i + 1]
            x = sequence_features(seq, seq.index, mean, std)

            if x is None:
                continue

            y_targets = []

            for h in HORIZONS:
                y_targets.append((float(s.iloc[i + h]) - mean) / std)

            X_seq.append(x)
            Y.append(y_targets)
            D_ids.append(info["disease_id"])
            L_ids.append(info["location_id"])

    return (
        np.asarray(X_seq, dtype="float32"),
        np.asarray(Y, dtype="float32"),
        np.asarray(D_ids, dtype="int32").reshape(-1, 1),
        np.asarray(L_ids, dtype="int32").reshape(-1, 1)
    )


def build_dl_eval_inputs(keys):
    X_seq, D_ids, L_ids = [], [], []
    valid_meta, invalid_meta = [], []

    for key in keys:
        info = SERIES_DATA[key]

        train_s = info["train"]
        raw_full = info["raw_full"]
        y_test = raw_full.reindex(test_weeks)

        available = pd.Series(index=full_weeks, dtype=float)
        available.loc[train_s.index] = train_s.values

        mean = info["mean"]
        std = info["std"]

        for origin in test_weeks:
            if origin in y_test.index and pd.notna(y_test.loc[origin]):
                available.loc[origin] = float(y_test.loc[origin])

            seq_dates = pd.date_range(end=origin, periods=SEQ_LEN, freq=WEEK_FREQ)
            seq_vals = available.reindex(seq_dates).ffill()

            x = sequence_features(seq_vals, seq_dates, mean, std)

            meta = {
                "key": key,
                "disease": info["disease"],
                "location": info["location"],
                "origin": origin,
                "mean": mean,
                "std": std
            }

            if x is None:
                invalid_meta.append(meta)
            else:
                X_seq.append(x)
                D_ids.append(info["disease_id"])
                L_ids.append(info["location_id"])
                valid_meta.append(meta)

    return (
        np.asarray(X_seq, dtype="float32"),
        np.asarray(D_ids, dtype="int32").reshape(-1, 1),
        np.asarray(L_ids, dtype="int32").reshape(-1, 1),
        valid_meta,
        invalid_meta
    )


def build_global_gru_model(num_diseases, num_locations):
    seq_in = layers.Input(shape=(SEQ_LEN, 3), name="seq")
    disease_in = layers.Input(shape=(1,), dtype="int32", name="disease_id")
    location_in = layers.Input(shape=(1,), dtype="int32", name="location_id")

    x = layers.GRU(64, return_sequences=False)(seq_in)
    x = layers.Dropout(0.20)(x)

    d_emb = layers.Embedding(num_diseases, 4)(disease_in)
    d_emb = layers.Flatten()(d_emb)

    l_emb = layers.Embedding(num_locations, 8)(location_in)
    l_emb = layers.Flatten()(l_emb)

    x = layers.Concatenate()([x, d_emb, l_emb])
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.15)(x)
    x = layers.Dense(32, activation="relu")(x)

    out = layers.Dense(MAX_H, activation="linear")(x)

    model = models.Model(
        inputs=[seq_in, disease_in, location_in],
        outputs=out
    )

    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss=losses.Huber(delta=1.0),
        metrics=["mae"]
    )

    return model


def dl_predictions_to_long(valid_meta, pred_norm, invalid_meta=None):
    if invalid_meta is None:
        invalid_meta = []

    rows = []

    for meta, pred_vec in zip(valid_meta, pred_norm):
        key = meta["key"]
        info = SERIES_DATA[key]
        y_test = info["raw_full"].reindex(test_weeks)

        origin = meta["origin"]
        mean = meta["mean"]
        std = meta["std"]

        for h in HORIZONS:
            target = origin + pd.Timedelta(weeks=h)

            if target not in y_test.index:
                continue

            actual = y_test.loc[target]

            pred = float(pred_vec[h - 1] * std + mean)

            rows.append({
                "origin": origin,
                "target": target,
                "disease": meta["disease"],
                "location": meta["location"],
                "h": int(h),
                "y": float(actual) if pd.notna(actual) else np.nan,
                "pred": pred,
                "model": DL_GRU_NAME
            })

    for meta in invalid_meta:
        key = meta["key"]
        info = SERIES_DATA[key]
        y_test = info["raw_full"].reindex(test_weeks)
        origin = meta["origin"]

        for h in HORIZONS:
            target = origin + pd.Timedelta(weeks=h)

            if target not in y_test.index:
                continue

            actual = y_test.loc[target]

            rows.append({
                "origin": origin,
                "target": target,
                "disease": meta["disease"],
                "location": meta["location"],
                "h": int(h),
                "y": float(actual) if pd.notna(actual) else np.nan,
                "pred": np.nan,
                "model": DL_GRU_NAME
            })

    return pd.DataFrame(rows)


if RUN_DL:
    print("\nTraining DL Global GRU final-test model...")

    tf.keras.backend.clear_session()

    X_seq, Y, D_ids, L_ids = build_dl_train_samples(ALL_KEYS)

    print("DL train samples:", X_seq.shape, Y.shape)

    dl_model = build_global_gru_model(
        num_diseases=len(DISEASE_TO_ID),
        num_locations=len(LOCATION_TO_ID)
    )

    cb = [
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE_DL,
            restore_best_weights=True
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-5
        )
    ]

    hist = dl_model.fit(
        [X_seq, D_ids, L_ids],
        Y,
        validation_split=0.15,
        epochs=EPOCHS_DL,
        batch_size=BATCH_SIZE,
        shuffle=True,
        callbacks=cb,
        verbose=1
    )

    X_eval, D_eval, L_eval, valid_meta, invalid_meta = build_dl_eval_inputs(ALL_KEYS)

    pred_norm = dl_model.predict(
        [X_eval, D_eval, L_eval],
        batch_size=256,
        verbose=0
    )

    dl_long = dl_predictions_to_long(valid_meta, pred_norm, invalid_meta)

    dl_status = pd.DataFrame([{
        "model": DL_GRU_NAME,
        "n_train_samples": len(X_seq),
        "epochs_run": len(hist.history["loss"]),
        "final_loss": float(hist.history["loss"][-1]),
        "final_val_loss": float(hist.history["val_loss"][-1])
    }])
else:
    dl_long = pd.DataFrame()
    dl_status = pd.DataFrame()


# ============================================================
# 10. Combine base predictions
# ============================================================

base_long_parts = [
    sarima_long,
    rf_local_long,
    rf_pooled_long,
]

if RUN_DL:
    base_long_parts.append(dl_long)

base_long = pd.concat(base_long_parts, ignore_index=True)

base_long = base_long[
    base_long["model"].isin(SELECTED_BASE_MODELS)
].copy()

print("\nBase long predictions:")
print(base_long.shape)
display(base_long.head())


# ============================================================
# 11. Mean and median ensembles
# ============================================================

def build_ensembles(base_long, selected_models):
    keys = ["origin", "target", "disease", "location", "h", "y"]

    wide = (
        base_long
        .pivot_table(
            index=keys,
            columns="model",
            values="pred",
            aggfunc="first"
        )
        .reset_index()
    )

    missing_models = [m for m in selected_models if m not in wide.columns]

    if missing_models:
        raise ValueError(
            f"Cannot build 5-model ensembles. Missing models: {missing_models}"
        )

    wide["n_available_models"] = wide[selected_models].notna().sum(axis=1)

    ensemble_valid = wide[wide["n_available_models"] == len(selected_models)].copy()

    mean_rows = ensemble_valid[keys].copy()
    mean_rows["pred"] = ensemble_valid[selected_models].mean(axis=1)
    mean_rows["model"] = ENSEMBLE_MEAN_NAME

    median_rows = ensemble_valid[keys].copy()
    median_rows["pred"] = ensemble_valid[selected_models].median(axis=1)
    median_rows["model"] = ENSEMBLE_MEDIAN_NAME

    component_count = (
        wide
        .groupby(["disease", "location", "h"], as_index=False)
        .agg(
            total_rows=("n_available_models", "size"),
            rows_with_all_5=("n_available_models", lambda x: int((x == len(selected_models)).sum())),
            min_available_models=("n_available_models", "min"),
            max_available_models=("n_available_models", "max")
        )
    )

    return pd.concat([mean_rows, median_rows], ignore_index=True), component_count


ensemble_long, ensemble_component_counts = build_ensembles(
    base_long,
    SELECTED_BASE_MODELS
)

all_long = pd.concat([base_long, ensemble_long], ignore_index=True)

print("\nEnsemble component count:")
display(ensemble_component_counts.head())

print("\nAll long predictions including ensembles:")
print(all_long.shape)
display(all_long.head())


# ============================================================
# 12. Metrics and audit
# ============================================================

def build_country_h(long_df):
    rows = []

    long_df = long_df.merge(
        mase_scales,
        on=["disease", "location"],
        how="left"
    )

    for (model, disease, location, h), sub in long_df.groupby(["model", "disease", "location", "h"]):
        sub = sub.copy()
        sub = sub[sub["y"].notna()]
        sub = sub[sub["pred"].notna()]
        sub = sub[sub["mase_scale"].notna()]

        if len(sub) == 0:
            continue

        m_mae = mae_np(sub["y"].values, sub["pred"].values)
        m_rmse = rmse_np(sub["y"].values, sub["pred"].values)
        scale = float(sub["mase_scale"].iloc[0])
        m_mase = mase_from_mae(m_mae, scale)

        rows.append({
            "disease": disease,
            "location": location,
            "h": int(h),
            "model": model,
            "MAE": float(m_mae),
            "RMSE": float(m_rmse),
            "MASE": float(m_mase),
            "n": int(len(sub))
        })

    return pd.DataFrame(rows).sort_values(["disease", "location", "h", "model"]).reset_index(drop=True)


final_country_h = build_country_h(all_long)

expected_by_model = expected_points.merge(
    final_country_h[["disease", "location", "model"]].drop_duplicates(),
    on=["disease", "location"],
    how="inner"
)

scored = (
    final_country_h
    .groupby(["model", "disease", "location", "h"], as_index=False)
    .agg(scored_points=("n", "sum"))
)

n_audit = expected_by_model.merge(
    scored,
    on=["model", "disease", "location", "h"],
    how="left"
)

n_audit["scored_points"] = n_audit["scored_points"].fillna(0).astype(int)
n_audit["missing_due_to_model_or_pred_nan"] = (
    n_audit["expected_actual_points"] - n_audit["scored_points"]
)

horizon_summary = (
    final_country_h
    .groupby(["disease", "h", "model"], as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        MASE=("MASE", "mean"),
        n_countries=("location", "nunique"),
        n_points=("n", "sum")
    )
    .sort_values(["disease", "h", "MASE", "MAE"])
    .reset_index(drop=True)
)

country_summary = (
    final_country_h
    .groupby(["disease", "location", "model"], as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        MASE=("MASE", "mean"),
        n_points=("n", "sum")
    )
    .sort_values(["disease", "location", "MASE", "MAE"])
    .reset_index(drop=True)
)

winners_by_h = (
    horizon_summary
    .sort_values(["disease", "h", "MASE", "MAE"])
    .groupby(["disease", "h"], group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

top5_by_h = (
    horizon_summary
    .sort_values(["disease", "h", "MASE", "MAE"])
    .groupby(["disease", "h"], group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

common6_country_h = final_country_h[final_country_h["location"].isin(COMMON6)].copy()

common6_horizon_summary = (
    common6_country_h
    .groupby(["disease", "h", "model"], as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        MASE=("MASE", "mean"),
        n_countries=("location", "nunique"),
        n_points=("n", "sum")
    )
    .sort_values(["disease", "h", "MASE", "MAE"])
    .reset_index(drop=True)
)

common6_winners_by_h = (
    common6_horizon_summary
    .sort_values(["disease", "h", "MASE", "MAE"])
    .groupby(["disease", "h"], group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

audit_agg = (
    n_audit
    .groupby(["model", "disease", "h"], as_index=False)
    .agg(
        expected_actual_points=("expected_actual_points", "sum"),
        scored_points=("scored_points", "sum"),
        missing_due_to_model_or_pred_nan=("missing_due_to_model_or_pred_nan", "sum")
    )
    .sort_values(["model", "disease", "h"])
    .reset_index(drop=True)
)

print("\nFinal test audit:")
display(audit_agg)

print("\nFinal horizon summary:")
display(horizon_summary)

print("\nFinal winners by horizon:")
display(winners_by_h)

print("\nCommon6 horizon summary:")
display(common6_horizon_summary)

print("\nCommon6 winners by horizon:")
display(common6_winners_by_h)


# ============================================================
# 13. Save outputs
# ============================================================

ari_files_used.to_csv(FINAL_OUT_DIR / "ari_snapshot_files_used.csv", index=False)
ili_files_used.to_csv(FINAL_OUT_DIR / "ili_snapshot_files_used.csv", index=False)

expected_points.to_csv(FINAL_OUT_DIR / "finaltest_expected_points_2024_latest.csv", index=False)
mase_scales.to_csv(FINAL_OUT_DIR / "finaltest_mase_scales_2014_2023.csv", index=False)

base_long.to_csv(FINAL_OUT_DIR / "finaltest_base_predictions_long.csv", index=False)
ensemble_long.to_csv(FINAL_OUT_DIR / "finaltest_ensemble_predictions_long.csv", index=False)
all_long.to_csv(FINAL_OUT_DIR / "finaltest_all_predictions_long.csv", index=False)

final_country_h.to_csv(FINAL_OUT_DIR / "finaltest_country_h.csv", index=False)
horizon_summary.to_csv(FINAL_OUT_DIR / "finaltest_horizon_summary.csv", index=False)
country_summary.to_csv(FINAL_OUT_DIR / "finaltest_country_summary.csv", index=False)

top5_by_h.to_csv(FINAL_OUT_DIR / "finaltest_top5_by_h.csv", index=False)
winners_by_h.to_csv(FINAL_OUT_DIR / "finaltest_winners_by_h.csv", index=False)

common6_horizon_summary.to_csv(FINAL_OUT_DIR / "finaltest_common6_horizon_summary.csv", index=False)
common6_winners_by_h.to_csv(FINAL_OUT_DIR / "finaltest_common6_winners_by_h.csv", index=False)

n_audit.to_csv(FINAL_OUT_DIR / "finaltest_n_audit.csv", index=False)
audit_agg.to_csv(FINAL_OUT_DIR / "finaltest_n_audit_aggregated.csv", index=False)

ensemble_component_counts.to_csv(FINAL_OUT_DIR / "finaltest_ensemble_component_counts.csv", index=False)

pd.DataFrame(model_status_rows).to_csv(FINAL_OUT_DIR / "finaltest_model_status_sarima_autoarima.csv", index=False)

if RUN_DL:
    dl_status.to_csv(FINAL_OUT_DIR / "finaltest_model_status_dl.csv", index=False)

print("\nSaved final-test outputs to:")
print(FINAL_OUT_DIR)

print("\nDone.")

ARI loaded: (7686, 5)
ILI loaded: (11987, 5)

ARI files used:


,snapshot_date,file
0,2024-10-11,2024-10-11-ARI_incidence.csv
1,2024-10-18,2024-10-18-ARI_incidence.csv
2,2024-10-25,2024-10-25-ARI_incidence.csv
3,2024-11-08,2024-11-08-ARI_incidence.csv
4,2024-11-15,2024-11-15-ARI_incidence.csv


,snapshot_date,file
54,2025-11-21,2025-11-21-ARI_incidence.csv
55,2025-11-28,2025-11-28-ARI_incidence.csv
56,2025-12-05,2025-12-05-ARI_incidence.csv
57,2025-12-12,2025-12-12-ARI_incidence.csv
58,2025-12-19,2025-12-19-ARI_incidence.csv



ILI files used:


,snapshot_date,file
0,2024-10-11,2024-10-11-ILI_incidence.csv
1,2024-10-18,2024-10-18-ILI_incidence.csv
2,2024-10-25,2024-10-25-ILI_incidence.csv
3,2024-11-08,2024-11-08-ILI_incidence.csv
4,2024-11-15,2024-11-15-ILI_incidence.csv


...


Common6 horizon summary:


,disease,h,model,MAE,RMSE,MASE,n_countries,n_points
0,ARI,1,ensemble_mean_5,90.943531,129.528136,0.437441,6,591
1,ARI,1,ensemble_median_5,91.955605,129.698853,0.442718,6,591
2,ARI,1,DL_GlobalGRU_all_infections_all_countries,94.063769,131.844275,0.458638,6,591
3,ARI,1,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",96.947553,135.938739,0.462649,6,591
4,ARI,1,rf_global_all_infections_all_countries,99.978547,139.430756,0.476647,6,591
5,ARI,1,autoARIMA,99.267819,137.820053,0.476907,6,591
6,ARI,1,RandomForest(lags=4),112.363564,155.460664,0.549910,6,591
7,ARI,2,DL_GlobalGRU_all_infections_all_countries,115.800395,159.614687,0.561271,6,585
8,ARI,2,ensemble_mean_5,119.675180,166.711379,0.574992,6,585
9,ARI,2,ensemble_median_5,122.602430,169.931583,0.589695,6,585



Common6 winners by horizon:


,disease,h,model,MAE,RMSE,MASE,n_countries,n_points
0,ARI,1,ensemble_mean_5,90.943531,129.528136,0.437441,6,591
1,ARI,2,DL_GlobalGRU_all_infections_all_countries,115.800395,159.614687,0.561271,6,585
2,ARI,3,DL_GlobalGRU_all_infections_all_countries,127.423516,174.448743,0.621027,6,579
3,ARI,4,DL_GlobalGRU_all_infections_all_countries,134.239504,184.831754,0.661633,6,573
4,ILI,1,ensemble_median_5,17.901387,33.348414,0.642640,6,591
5,ILI,2,ensemble_mean_5,24.445075,48.253526,0.899550,6,585
6,ILI,3,rf_global_all_infections_all_countries,31.494003,59.478580,1.118209,6,579
7,ILI,4,rf_global_all_infections_all_countries,34.419638,63.788240,1.226477,6,573



Saved final-test outputs to:
C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025

Done.
